# PT-06 — Evaluation Comparative du Post-Training

**Serie** : Post-Training SOTA 2024-2025 (Epic #1742)
**Dernier livrable** de la serie
**Pre-requis** : PT-01 a PT-05 (tous merges)
**Objectifs pedagogiques** :
1. Synthetiser les 4 techniques de post-training etudiees (SFT, DPO, GRPO, RLVR)
2. Comparer les metriques cibles : cout compute, qualite, data requirements, complexite
3. Definir un framework de decision pour choisir la bonne technique
4. Identifier les perspectives de recherche 2025-2026
5. Proposer un pipeline post-training optimal pour differents scenarios

**References cles** :
- Rafailov et al., "Direct Preference Optimization" (2023) — DPO
- Shao et al., "DeepSeekMath" (2024) — GRPO
- Deepseek-AI, "DeepSeek-R1" (2025) — RLVR
- Ouyang et al., "Training language models to follow instructions with human feedback" (InstructGPT, 2022)

## 1. Recap de la serie Post-Training

Au cours de cette serie (PT-01 a PT-05), nous avons etudie les techniques majeures du post-training 2024-2025 :

| Notebook | Technique | Idee cle |
|----------|-----------|----------|
| PT-01 | Introduction | Landscape RLHF, taxonomie des approches |
| PT-02 | SFT (LoRA) | Point de depart obligatoire, fine-tuning supervise |
| PT-03 | DPO | Preference learning sans RL, paires chosen/rejected |
| PT-04 | GRPO | RL sans critic, avantage intra-group |
| PT-05 | RLVR | GRPO avec rewards verifiables (zero bruit) |

### L'evolution conceptuelle

```
SFT (supervise)
  |
  v
DPO (preferences humaines = "qu'est-ce qui est meilleur ?")
  |
  v
GRPO (rewards = "est-ce que c'est bon ?")
  |
  v
RLVR (verifiable = "est-ce que c'est EXACT ?")
```

Chaque etape **affine** le modele de maniere complementaire. Le pipeline optimal n'est pas une seule technique mais une **combinaison**.

### Exercice 1 : Creer un tableau comparatif personnalise

L'objectif de cet exercice est de creer un tableau comparatif qui inclut une dimension supplementaire non couverte dans la section 2 : la "facilite de debug" de chaque technique (facile, moyen, difficile).

**Contexte** : Le tableau de la section 2 couvre 8 criteres techniques. Vous allez ajouter des criteres pratiques (debug, reproductibilite, accessibilite pour un debutant).

**Indices** :
- Debug : SFT (facile = loss standard), DPO (moyen = 2 modeles), GRPO (difficile = sampling + reward), RLVR (moyen = verifier exact)
- Reproductibilite : depend du seed, de la temperature, du sampling
- Utilisez `pandas.DataFrame` avec une colonne par critere supplementaire

In [ ]:
# Exercice 1 : Tableau comparatif enrichi
# TODO etudiant : ajoutez les colonnes Debug, Reproductibilite, Accessibilite

# extended_comparison = {
#     "Technique": ["SFT", "DPO", "GRPO", "RLVR"],
#     "Facilite debug": [None, None, None, None],  # TODO etudiant
#     "Reproductibilite": [None, None, None, None],  # TODO etudiant
#     "Accessibilite debutant": [None, None, None, None],  # TODO etudiant
# }
# df = pd.DataFrame(extended_comparison)
# print(df.to_string(index=False))

print("Exercice a completer : tableau comparatif enrichi")

## 2. Tableau comparatif des 4 techniques

### 2.1 Comparaison technique

| Critere | SFT | DPO | GRPO | RLVR |
|---------|-----|-----|------|------|
| **Type d'apprentissage** | Supervise | Preference | RL (sans critic) | RL (reward exact) |
| **Donnees necessaires** | Input-Output paires | Paires chosen/rejected | Prompts + reward fn | Prompts + verifier |
| **Cout compute (relatif)** | 1x | 1.5x | 2x | 2x |
| **Memoire GPU (relatif)** | 1x | 2x (ref model) | 2x (G completions) | 2x (G completions) |
| **Hyperparametres cles** | lr, epochs | beta, lr | beta, G, epsilon | beta, G, lr |
| **Reward hacking** | N/A | Possible (biais data) | Possible | **Impossible** |
| **Domaine optimal** | General | Alignement style | General RL | Math, code, puzzles |
| **Implementation TRL** | SFTTrainer | DPOTrainer | GRPOTrainer | GRPOTrainer + verifier |

### 2.2 Comparaison des hyperparametres

| Parametre | SFT | DPO | GRPO | RLVR |
|-----------|-----|-----|------|------|
| Learning rate | 2e-4 | 5e-7 | 1e-5 | 5e-6 |
| Beta (KL) | N/A | 0.1 | 0.04 | 0.04 |
| Group size | N/A | N/A | 4-16 | 4-16 |
| Batch size | 4-16 | 4-16 | 1-4 | 1-4 |
| Max sequence | 512 | 512 | 256-512 | 256-512 |
| Epochs | 3-5 | 1-3 | 1-3 | 1-3 |

### 2.3 Cout data

| Technique | Volume data min. | Type | Cout annotation |
|-----------|-------------------|------|-----------------|
| SFT | 1K-10K | Input-Output | Eleve (humain) |
| DPO | 1K-10K paires | Preferences | Tres eleve (comparaison humaine) |
| GRPO | 100-1K prompts | + Reward function | Faible (fonction a ecrire) |
| RLVR | 100-1K prompts | + Verifier exact | Tres faible (code deterministe) |

In [1]:
import pandas as pd

# Tableau comparatif interactif
comparison_data = {
    "Technique": ["SFT", "DPO", "GRPO", "RLVR"],
    "Notebook": ["PT-02", "PT-03", "PT-04", "PT-05"],
    "Learning Rate": ["2e-4", "5e-7", "1e-5", "5e-6"],
    "Beta (KL)": ["N/A", "0.1", "0.04", "0.04"],
    "GPU Memory (relatif)": ["1x", "2x", "2x", "2x"],
    "Data Min": ["1K samples", "1K pairs", "100 prompts", "100 prompts"],
    "Reward Type": ["N/A (labels)", "N/A (pairs)", "Heuristic", "Exact (zero-noise)"],
    "TRL Class": ["SFTTrainer", "DPOTrainer", "GRPOTrainer", "GRPOTrainer + verifier"],
    "Risk Hacking": ["N/A", "Possible", "Possible", "Impossible"],
}

df = pd.DataFrame(comparison_data)
df = df.set_index("Technique")
print("Tableau Comparatif Post-Training SOTA 2024-2025")
print("=" * 80)
print(df.to_string())
print()
print(f"Total notebooks dans la serie : 6 (PT-01 a PT-06)")
print(f"Modele utilise pour les demos : Qwen2.5-0.5B-Instruct")
print(f"Framework : HuggingFace TRL + PEFT (LoRA)")

Tableau Comparatif Post-Training SOTA 2024-2025
          Notebook Learning Rate Beta (KL) GPU Memory (relatif)     Data Min         Reward Type               TRL Class Risk Hacking
Technique                                                                                                                            
SFT          PT-02          2e-4       N/A                   1x   1K samples        N/A (labels)              SFTTrainer          N/A
DPO          PT-03          5e-7       0.1                   2x     1K pairs         N/A (pairs)              DPOTrainer     Possible
GRPO         PT-04          1e-5      0.04                   2x  100 prompts           Heuristic             GRPOTrainer     Possible
RLVR         PT-05          5e-6      0.04                   2x  100 prompts  Exact (zero-noise)  GRPOTrainer + verifier   Impossible

Total notebooks dans la serie : 6 (PT-01 a PT-06)
Modele utilise pour les demos : Qwen2.5-0.5B-Instruct
Framework : HuggingFace TRL + PEFT (LoRA)


## 3. Framework de decision

### Arbre de decision pour choisir la bonne technique

```
Question : Quel est votre objectif ?
│
├── Alignement sur un style/format specifique
│   └── SFT (PT-02)
│
├── Correction de comportements indesirables avec donnees humaines
│   └── DPO (PT-03)
│
├── Optimisation d'une metric objective (general)
│   └── GRPO (PT-04)
│
└── Domaine avec reponse exacte verifiable (math, code)
    └── RLVR (PT-05)
```

### Recommandations par scenario

| Scenario | Pipeline recommande | Raison |
|----------|---------------------|--------|
| Chatbot d'entreprise | SFT → DPO | Style specifique + alignement preferences |
| Modele mathematique | SFT → RLVR | Raisonnement exact, pas de subjectivite |
| Code assistant | SFT → RLVR | Execution verifiable (tests unitaires) |
| Redaction creative | SFT → DPO | Subjective, preferences humaines necessaires |
| Modele generaliste | SFT → DPO → GRPO | Maximum d'alignement |
| Reasoning emergent | SFT → RLVR | C'est le chemin Deepseek-R1 |

### Quand NE PAS utiliser chaque technique

- **SFT seul** : si on veut un comportement qui n'existe pas dans les donnees d'entrainement
- **DPO** : si on n'a PAS de donnees de preference humaine (couteux a annoter)
- **GRPO heuristique** : si on a un domaine avec verifier exact (utiliser RLVR a la place)
- **RLVR** : si le domaine n'a PAS de reponse exacte (ex : generation creative)

### Exercice 2 : Visualiser les compromis compute vs qualite

L'objectif de cet exercice est de creer un scatter plot montrant le compromis entre le cout compute (axe X) et la qualite estimee (axe Y) pour les 4 techniques, avec la taille des points proportionnelle au cout d'annotation.

**Contexte** : Les tableaux de la section 2 fournissent des donnees quantitatives. Vous allez les visualiser ensemble.

**Indices** :
- Axe X : GPU-heures (estimes dans la section 2)
- Axe Y : Accuracy GSM8K estimee (tableau de la section 3)
- Taille des points : heures d'annotation (proportionnel)
- Utilisez `plt.scatter` avec le parametre `s` pour la taille
- Ajoutez des etiquettes pour chaque point

In [ ]:
# Exercice 2 : Scatter plot compute vs qualite
# TODO etudiant : creez le scatter plot avec les 4 techniques

# techniques = ["SFT", "DPO", "GRPO", "RLVR"]
# gpu_hours = [0.5, 0.8, 1.2, 1.3]
# accuracy = [35, 40, 45, 55]
# annotation_hours = [40, 120, 120, 0]

# fig, ax = plt.subplots(figsize=(8, 5))
# # TODO etudiant : plt.scatter(gpu_hours, accuracy, s=annotation_sizes)
# # Ajoutez les etiquettes pour chaque point
# plt.show()

print("Exercice a completer : scatter plot compute vs qualite")

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Simulation des couts compute relatifs (estimes pour Qwen2.5-0.5B, 1K samples, 3 epochs)
techniques = ["SFT", "DPO", "GRPO", "RLVR"]
gpu_hours = [0.5, 0.8, 1.2, 1.3]  # Heures GPU RTX 3070
memory_gb = [2.0, 3.5, 4.0, 4.0]  # Go VRAM

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63"]

# GPU hours
bars1 = ax1.bar(techniques, gpu_hours, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Heures GPU (RTX 3070, estim.)")
ax1.set_title("Cout Compute Relatif")
ax1.set_ylim(0, max(gpu_hours) * 1.3)
for bar, val in zip(bars1, gpu_hours):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, f"{val}h", ha="center", fontsize=10)

# Memory
bars2 = ax2.bar(techniques, memory_gb, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("VRAM requise (Go)")
ax2.set_title("Memoire GPU Relatif")
ax2.set_ylim(0, max(memory_gb) * 1.4)
for bar, val in zip(bars2, memory_gb):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f"{val} Go", ha="center", fontsize=10)

plt.suptitle("Post-Training SOTA 2024-2025 — Cout Compute vs Memoire (Qwen2.5-0.5B, 1K samples, 3 epochs)", fontsize=11)
plt.tight_layout()
plt.savefig("pt06_cost_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure sauvegardee : pt06_cost_comparison.png")

Figure sauvegardee : pt06_cost_comparison.png


C:\Users\jsboi\AppData\Local\Temp\claude\ipykernel_459428\4251458305.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Simulation qualitative des resultats attendus par technique
# (Estimes, pas de claim BEATS — terrain pedagogique)

results = {
    "Technique": ["Base (no FT)", "SFT", "DPO", "GRPO", "RLVR"],
    "Accuracy GSM8K (%)": [15, 35, 40, 45, 55],
    "Coherence (1-5)": [2.5, 3.5, 4.0, 3.8, 4.2],
    "Instruction Following (1-5)": [2.0, 4.0, 4.5, 4.0, 4.0],
    "Reasoning Depth (1-5)": [1.5, 2.5, 3.0, 3.5, 4.5],
    "Style Control (1-5)": [1.0, 4.5, 4.0, 3.0, 3.0],
}

df_results = pd.DataFrame(results)
df_results = df_results.set_index("Technique")
print("Resultats attendus (Qwen2.5-0.5B, estimation pedagogique)")
print("=" * 80)
print(df_results.to_string())
print()
print("Note : ces chiffres sont des ESTIMATIONS pedagogiques.")
print("Pas de claim BEATS quantitatif (sample insuffisant pour multi-seed).")
print("Pour des resultats rigoureux : voir papers de reference (DPO, GRPO, RLVR).")

Resultats attendus (Qwen2.5-0.5B, estimation pedagogique)
              Accuracy GSM8K (%)  Coherence (1-5)  Instruction Following (1-5)  Reasoning Depth (1-5)  Style Control (1-5)
Technique                                                                                                                 
Base (no FT)                  15              2.5                          2.0                    1.5                  1.0
SFT                           35              3.5                          4.0                    2.5                  4.5
DPO                           40              4.0                          4.5                    3.0                  4.0
GRPO                          45              3.8                          4.0                    3.5                  3.0
RLVR                          55              4.2                          4.0                    4.5                  3.0

Note : ces chiffres sont des ESTIMATIONS pedagogiques.
Pas de claim BEATS quanti

## 4. Le pipeline post-training optimal (synthese)

### Le pipeline en 4 etapes (recommandation 2025)

```
Etape 1 : SFT (obligatoire)
  - Pourquoi : le modele de base ne suit pas d'instructions
  - Cout : modere (1K-10K examples)
  - Gain : instruction following de base

Etape 2 : DPO (recommande si budget annotation)
  - Pourquoi : alignement fin sur les preferences humaines
  - Cout : eleve (donnees de preference)
  - Gain : style, coherence, securite

Etape 3 : GRPO (optionnel, si reward function disponible)
  - Pourquoi : optimiser une metric objective
  - Cout : modere (compute + reward function)
  - Gain : performance sur la metric ciblee

Etape 4 : RLVR (si domaine verifiable)
  - Pourquoi : raisonner de maniere fiable
  - Cout : faible (verifier = code deterministe)
  - Gain : emergence de raisonnement, accuracy math/code
```

### Ce que Deepseek-R1 a fait differemment

Deepseek-R1 a **saute l'etape DPO** et est passe directement SFT → RLVR. Pourquoi ?
- Pas de donnees de preference humaine a grande echelle
- Le reward verifiable (math) etait suffisant pour guider l'apprentissage
- Resultat : emergence de chain-of-thought **sans supervision humaine**

Leçon : si votre domaine a un verifier exact, DPO est **optionnel**. RLVR seul suffit pour le raisonnement.

## 5. Perspectives de recherche 2025-2026

### 5.1 Techniques emergentes

| Technique | Paper | Ideee | Statut |
|-----------|-------|-------|--------|
| **ORPO** | Hong et al. (2024) | Merge SFT + DPO en une seule etape | Production-ready |
| **KTO** | Ethayarajh et al. (2024) | DPO sans paires (points individuels) | TRL integre |
| **NCA** | Chen et al. (2024) | Noise Contrastive Alignment | Experimental |
| **RLOO** | Ahmadian et al. (2024) | Leave-one-out baseline pour REINFORCE | TRL integre |
| **Online DPO** | Guo et al. (2024) | DPO avec generation en ligne | Actif |
| **Self-play RL** | Meta (2025) | Le modele genere ses propres preferences | Recherche |

### 5.2 Tendances cles

1. **Elimination du reward model** : DPO, GRPO, RLVR suppriment le besoin d'entrainer un RM separe. Tendance qui s'accelere.

2. **Process reward > Outcome reward** : verifier chaque etape du raisonnement (pas seulement la reponse finale). Plus precis, mais plus complexe.

3. **Scaling laws du RL** : le compute RL est de plus en plus efficace. Deepseek-R1 a montre que RL pur > SFT+RL sur le raisonnement.

4. **Constitutional AI** : le modele apprend de ses propres outputs via des principles (Anthropic). Evolue vers self-alignment.

5. **Multimodal post-training** : etendre SFT/DPO/GRPO aux modeles vision, audio, video. Challenge : definir des rewards verifiables pour ces domaines.

### 5.3 Questions ouvertes

- **RLVR generalise** : peut-on definir des verifiers pour des domaines non-mathematiques (creation, argumentation) ?
- **Emergence vs scale** : le phenomene d'emergence du raisonnement est-il robuste a travers les tailles de modeles ?
- **Data efficiency** : combien de prompts sont reellement necessaires pour RLVR ? 100 ? 1 000 ? 10 000 ?
- **Safety** : comment garantir que le RL n'introduit pas de comportements dangereux lors de l'exploration ?

## 6. Conclusion de la serie Post-Training

### Ce que nous avons couvert (PT-01 a PT-06)

| Notebook | Contribution clee |
|----------|-------------------|
| PT-01 | **Cartographie** du paysage post-training 2024-2025 |
| PT-02 | **SFT LoRA** : le fondement, fine-tuning param-efficace |
| PT-03 | **DPO** : preference learning sans RL, Bradley-Terry |
| PT-04 | **GRPO** : RL sans critic, avantage intra-group |
| PT-05 | **RLVR** : rewards verifiables, emergence du raisonnement |
| **PT-06** | **Synthese** : comparaison, decision framework, perspectives |

### Les 3 lecons cles

1. **Le pipeline est cumulatif** : SFT → DPO → GRPO/RLVR. Chaque etape affine le modele. Ne pas sauter SFT.

2. **Le reward definit la technique** : si vous avez un verifier exact (math, code), utilisez RLVR. Si vous avez des preferences humaines, utilisez DPO. Sinon, GRPO avec reward heuristique.

3. **L'honnetete prime** : en pedagogie comme en recherche, rapporter honnetement les resultats. INCONCLUSIF > faux BEATS. 10 prompts ne font pas une evaluation rigoureuse.

### Pour aller plus loin

- **Practice** : re-executer les notebooks avec `LOAD_MODEL_AND_TRAIN = True` sur GPU
- **Papers** : lire les papers de reference (Rafailov 2023, Shao 2024, Deepseek-R1 2025)
- **TRL docs** : https://huggingface.co/docs/trl/ pour les dernieres integrations
- **Communaute** : HuggingFace Discord, r/LocalLLaMA pour les discussions pratiques

### Exercice 3 : Rediger un rapport de decision pour un cas concret

L'objectif de cet exercice est de choisir un scenario concret parmi ceux du framework de decision (section 3) et de rediger une recommandation technique argumentee de 10 lignes minimum.

**Contexte** : Le framework de decision propose 6 scenarios types. Vous allez en choisir un et justifier le pipeline recommande.

**Indices** :
- Identifiez le scenario (chatbot, code assistant, modele math, etc.)
- Listez les contraintes (budget GPU, donnees disponibles, delai)
- Justifiez chaque etape du pipeline recommande
- Mentionnez les risques et les alternatives ecartees

In [ ]:
# Exercice 3 : Rapport de decision
# TODO etudiant : choisissez un scenario et redigez votre recommandation

# SCENARIO = "Modele mathematique"  # ou "Chatbot d'entreprise", "Code assistant", etc.

# recommandation = """
# TODO etudiant : redigez votre recommandation technique ici (10 lignes min)
# - Pipeline recommande
# - Justification de chaque etape
# - Contraintes identifiees
# - Alternatives ecartees
# """

print("Exercice a completer : rapport de decision")